# Child model V&V: LBWSG artifact vs GBD

The upstream half of the chain: `child_lbwsg.ipynb` checks sim vs artifact; this notebook checks
that the artifact's LBWSG inputs match GBD, pulled fresh with `get_model_estimates`:

1. birth prevalence (maternal artifact's exposure key) vs GBD age group 164,
2. early/late-neonatal exposure (child artifact's exposure key) vs GBD age groups 2 and 3,
3. raw category relative risks (child artifact) vs the GBD LBWSG relative risk model (ME 10810).

Draw handling: the archived artifacts are mean-draw over `vi_globals.DRAW_COLUMNS` (250 draws),
while GBD 2023 returns 1000, so comparisons use the same 250-draw subset the build used.
Runs in the **artifact** environment (`source environment.sh -t artifact`) for GBD access;
the exposure pull takes ~5-10 minutes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger

logger.disable("vivarium_gbd_access")  # its cache-config DEBUG lines carry timestamps that would pollute rerun diffs

import vivarium.gbd_mapping as gbd_mapping

from vivarium_gbd_access.gbd.metadata import get_rei_modelable_entity_metadata
from vivarium_gbd_access.gbd.aggregation import aggregate_to_requested_location
from vivarium_gbd_access.shared_functions import get_model_estimates
from vivarium.artifact import Artifact
from vivarium_inputs import globals as vi_globals
from vivarium_inputs import utility_data

from lsff_utils import paths



In [ ]:
location = "nigeria"
vehicle = "rice"
data_source = paths.data_source(default=paths.MODEL_NUMBER)  # or "local" for this working tree


In [ ]:
artifact_dir = paths.artifact_path(
    paths.source_root(paths.CHILD_ARTIFACT_ROOT, data_source), location, vehicle
)
maternal_artifact_dir = paths.artifact_path(
    paths.source_root(paths.MATERNAL_ARTIFACT_ROOT, data_source), location, vehicle
)

rei_id = int(gbd_mapping.risk_factors.low_birth_weight_and_short_gestation.gbd_id)
me_ids = get_rei_modelable_entity_metadata(rei_id=rei_id, release_id=16)
location_id = utility_data.resolve_location(location.title())
RESIDUAL = vi_globals.EXTRA_RESIDUAL_CATEGORY["low_birth_weight_and_short_gestation"]
DRAW_COLS = list(vi_globals.DRAW_COLUMNS)
SEX = {1: "Male", 2: "Female"}
print(f"{data_source} {vehicle}/{location}\n  artifact: {artifact_dir}")
print(f"rei_id {rei_id}, location_id {location_id}, {len(DRAW_COLS)} draw columns, residual {RESIDUAL}")

In [ ]:
@aggregate_to_requested_location
def get_aggregated_model_estimates(
    modelable_entity_id: int | list[int],
    location_id: int | list[int] = [1, 2],
) -> pd.DataFrame:
    return get_model_estimates(
        modelable_entity_id=modelable_entity_id, 
        release_id=16,
        estimates="draws",
        location_id=location_id, 
        year_id=2021
    )

In [ ]:
# LBWSG birth prevalence is only estimated through 2021 in GBD 2023 (release 16), so both
# pulls use year 2021, mirroring the artifact builds. RRs come back for the Global location.
exp_raw = get_aggregated_model_estimates(
    modelable_entity_id=list(me_ids.modelable_entity_id[me_ids.draw_type == 'exposure']),
    location_id=location_id
    )

rr_raw = get_aggregated_model_estimates(
    modelable_entity_id=list(me_ids.modelable_entity_id[me_ids.draw_type == 'rr']),
    location_id=location_id
    )

exp_lookup = me_ids.loc[me_ids.draw_type == "exposure"].set_index("modelable_entity_id").categorical_parameter
exp_raw["parameter"] = exp_raw.modelable_entity_id.map(exp_lookup)

rr_raw = rr_raw.rename(columns={"category": "parameter"})

print(f"exposure {exp_raw.shape}, rr {rr_raw.shape}")

In [ ]:
def gbd_share(age_group_id):
    """Category shares: the artifact build's transform (drop residual, clip, normalize
    per draw within sex, then mean over the 250-draw subset)."""
    d = exp_raw[(exp_raw.age_group_id == age_group_id) & (exp_raw.parameter != RESIDUAL)]
    d = d.set_index(["sex_id", "parameter"])[DRAW_COLS]
    d = d.clip(lower=vi_globals.MINIMUM_EXPOSURE_VALUE)
    d = d / d.groupby("sex_id", observed=False).transform("sum")
    out = d.mean(axis=1).rename("gbd").reset_index()
    out["sex"] = out.sex_id.map(SEX)
    return out.set_index(["sex", "parameter"]).gbd


def report(cmp, name):
    log_ratio = np.log(cmp.gbd / cmp.artifact)
    print(f"{name}: max |log(GBD/artifact)| = {log_ratio.abs().max():.3g}, "
          f"exposure-weighted mean = {np.average(log_ratio.abs(), weights=cmp.artifact):.3g}")
    return cmp.reindex(log_ratio.abs().sort_values(ascending=False).index)

## 1. Birth prevalence: maternal artifact vs GBD (age group 164)

In [ ]:
mat = (
    Artifact(maternal_artifact_dir)
    .load("risk_factor.low_birth_weight_and_short_gestation.exposure")
    .reset_index()
    .set_index(["sex", "parameter"])
    .draw_0.rename("artifact")
)
cmp = pd.concat([gbd_share(164), mat], axis=1)
worst = report(cmp, "birth prevalence")

width = 0.4
for sex in ["Male", "Female"]:
    p = cmp.xs(sex, level="sex").sort_values("gbd").reset_index()
    plt.figure(dpi=1000, figsize=(6.4, 2.4))
    plt.bar(p.index, p.gbd, width=width, label="GBD")
    plt.bar(p.index + width, p.artifact, width=width, label="Artifact")
    plt.grid()
    plt.legend()
    plt.semilogy()
    plt.title(f"LBWSG birth prevalence, {sex}: GBD vs maternal artifact")

Agreement is within a few percent, and only in the rarest categories -- consistent with a
best-model-version refresh between the artifact build and this pull (the RR comparison below
shows the pipeline itself is deterministic to machine precision).

## 2. Early/late-neonatal exposure: child artifact vs GBD (age groups 2 and 3)

The child artifact's rows carry year 2023 (vivarium_inputs carries the 2021 estimates forward),
so values should match the 2021 pull.

In [ ]:
ce = Artifact(artifact_dir).load("risk_factor.low_birth_weight_and_short_gestation.exposure").reset_index()
for age_group_id, age_start, name in [(2, 0.0, "early neonatal"), (3, 0.019178, "late neonatal")]:
    ca = ce[np.isclose(ce.age_start, age_start)].set_index(["sex", "parameter"]).draw_0
    ca = (ca / ca.groupby("sex", observed=False).transform("sum")).rename("artifact")
    cmp = pd.concat([gbd_share(age_group_id), ca], axis=1)
    worst = report(cmp, f"{name} exposure")
print("\nlargest relative differences (all in the rarest categories):")
print(worst.head(6).round(8).to_string())

In [ ]:
p = cmp.xs("Female", level="sex").sort_values("gbd").reset_index()
plt.figure(dpi=1000, figsize=(6.4, 2.4))
plt.bar(p.index, p.gbd, width=width, label="GBD")
plt.bar(p.index + width, p.artifact, width=width, label="Artifact")
plt.grid()
plt.legend()
plt.semilogy()
plt.title("LBWSG late-neonatal exposure, Female: GBD vs child artifact")

## 3. Raw category relative risks: child artifact vs GBD

This is the key ingredient check for the interpolator scramble: the artifact's *raw* RR key
should match GBD exactly, so the fix only needs to touch the interpolator build, not the data.

In [ ]:
car = Artifact(artifact_dir).load("risk_factor.low_birth_weight_and_short_gestation.relative_risk").reset_index()
rr = rr_raw.copy()
rr["sex"] = rr.sex_id.map(SEX)
for age_group_id, age_start, name in [(2, 0.0, "early neonatal"), (3, 0.019178, "late neonatal")]:
    gbd_rr = (
        rr[rr.age_group_id == age_group_id]
        .set_index(["sex", "parameter"])[DRAW_COLS]
        .mean(axis=1)
        .rename("gbd")
    )
    art_rr = car[np.isclose(car.age_start, age_start)].set_index(["sex", "parameter"]).draw_0.rename("artifact")
    cmp = pd.concat([gbd_rr, art_rr], axis=1).dropna()
    max_err = np.log(cmp.gbd / cmp.artifact).abs().max()
    print(f"{name} RR ({len(cmp)} sex-category pairs): max |log(GBD/artifact)| = {max_err:.3g}")
    assert max_err < 1e-6, "raw RRs should match GBD exactly"
print("raw RRs match GBD exactly; the scramble is confined to the interpolator key")